# 05b - (Tahap 1) Eksplorasi dataset QA

In [1]:
import json, re, os
from collections import Counter, defaultdict
import random
import statistics

## 1. Konfigurasi

In [2]:
INPUT_PATH = "/kaggle/input/notebooks/muhammadnabilul/05-generating-context-awareqa/output_05a_v2/raw_records.jsonl"
OUTPUT_DIR = "output_05b_exploration"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
with open(INPUT_PATH, encoding="utf-8") as f:
    records = [json.loads(l) for l in f if l.strip()]
 
print(f"Total records: {len(records)}\n")
rng = random.Random(42)
report = {}  # akumulasi semua angka untuk diekspor sebagai satu laporan Bab 4

Total records: 12261



## 2. Distribusi

In [3]:
print("\n[Task Type]")
task_dist = Counter(r["task_type"] for r in records)
for k, v in task_dist.most_common():
    print(f"  {k:<30}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Subtask Type]")
subtask_dist = Counter(r["subtask_type"] for r in records)
for k, v in subtask_dist.most_common():
    print(f"  {k:<35}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Reasoning Source]")
reasoning_dist = Counter(r.get("reasoning_source", "?") for r in records)
for k, v in reasoning_dist.most_common():
    print(f"  {k:<25}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Generation Focus]")
focus_dist = Counter(r.get("generation_focus", "?") for r in records)
for k, v in focus_dist.most_common():
    print(f"  {k:<35}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Parse Status]")
parse_dist = Counter(r.get("parse_status", "?") for r in records)
for k, v in parse_dist.most_common():
    print(f"  {k:<20}: {v:>6} ({v/len(records)*100:.1f}%)")
 
report["taksonomi"] = {
    "task_type": dict(task_dist),
    "subtask_type": dict(subtask_dist),
    "reasoning_source": dict(reasoning_dist),
    "generation_focus": dict(focus_dist),
    "parse_status": dict(parse_dist),
}


[Task Type]
  qa                            :  11498 (93.8%)
  summary                       :    763 (6.2%)

[Subtask Type]
  factual_retrieval                  :   1778 (14.5%)
  definition_clarification           :   1492 (12.2%)
  comparative_analysis               :   1453 (11.9%)
  methodology_explanation            :   1444 (11.8%)
  source_navigation                  :   1357 (11.1%)
  causal_reasoning                   :   1337 (10.9%)
  trend_interpretation               :   1329 (10.8%)
  multi_source_synthesis             :   1308 (10.7%)
  document_overview                  :    763 (6.2%)

[Reasoning Source]
  single_paragraph         :   8887 (72.5%)
  multi_paragraph          :   1974 (16.1%)
  whole_section            :   1387 (11.3%)
                           :     13 (0.1%)

[Generation Focus]
  literal_reading                    :   3128 (25.5%)
  concept_understanding              :   2936 (23.9%)
  analytical_reasoning               :   2761 (22.5%)
  evidence_i

## 3. Cakupan korpus

In [4]:
doc_counts = Counter(r["document_title"] for r in records)
n_docs = len(doc_counts)
print(f"\n[Cakupan Dokumen]")
print(f"  Jumlah dokumen unik terwakili : {n_docs}")
print(f"  Record per dokumen  min/median/max : "
      f"{min(doc_counts.values())} / {statistics.median(doc_counts.values())} / {max(doc_counts.values())}")
 
print("\n  5 dokumen dengan record TERBANYAK:")
for doc, cnt in doc_counts.most_common(5):
    print(f"    {cnt:>4}  {doc[:60]}")
print("\n  5 dokumen dengan record TERSEDIKIT:")
for doc, cnt in sorted(doc_counts.items(), key=lambda x: x[1])[:5]:
    print(f"    {cnt:>4}  {doc[:60]}")
 
section_counts = Counter((r["document_title"], r["section_id"]) for r in records)
print(f"\n  Jumlah section unik terwakili : {len(section_counts)}")
 
report["cakupan_korpus"] = {
    "n_dokumen_unik": n_docs,
    "n_section_unik": len(section_counts),
    "record_per_dokumen_min": min(doc_counts.values()),
    "record_per_dokumen_median": statistics.median(doc_counts.values()),
    "record_per_dokumen_max": max(doc_counts.values()),
    "top5_terbanyak": doc_counts.most_common(5),
    "top5_tersedikit": sorted(doc_counts.items(), key=lambda x: x[1])[:5],
}


[Cakupan Dokumen]
  Jumlah dokumen unik terwakili : 29
  Record per dokumen  min/median/max : 51 / 313 / 1171

  5 dokumen dengan record TERBANYAK:
    1171  Statistik Kesejahteraan Rakyat 2023
    1092  Profil Kesehatan Ibu dan Anak 2024
     851  Statistik Kesejahteraan Rakyat 2024
     757  Statistik Indonesia 2025
     718  Statistik Indonesia 2023

  5 dokumen dengan record TERSEDIKIT:
      51  Infografis Statistik Kesejahteraan Rakyat 2025
     164  Indikator SDGs Kesejahteraan Rakyat 2025
     197  Profil Statistik Kesehatan 2025
     198  Laporan Indeks Khusus Penanganan Stunting (IKPS) 2022–2023
     206  Indeks Pembangunan Manusia 2022

  Jumlah section unik terwakili : 634


## 4. Panjang intruction & Response

In [5]:
inst_lens = [len(r["instruction"].split()) for r in records]
resp_lens = [len(r["response"].split()) for r in records]
cot_lens  = [len(r.get("cot", "").split()) for r in records]
span_counts = [len(r.get("context", [])) for r in records]
 
def stats(vals, label):
    print(f"\n[{label}]")
    print(f"  min={min(vals)} max={max(vals)} "
          f"mean={statistics.mean(vals):.1f} median={statistics.median(vals):.0f}")
    brackets = [(0,5),(5,10),(10,20),(20,50),(50,100),(100,200),(200,9999)]
    for lo, hi in brackets:
        n = sum(1 for v in vals if lo <= v < hi)
        if n: print(f"  [{lo:>4}\u2013{hi:>4} kata): {n:>6} ({n/len(vals)*100:.1f}%)")
    return {
        "min": min(vals), "max": max(vals),
        "mean": round(statistics.mean(vals), 2), "median": statistics.median(vals),
    }
 
stat_inst = stats(inst_lens, "Instruction (kata)")
stat_resp = stats(resp_lens, "Response (kata)")
stat_cot  = stats(cot_lens,  "CoT (kata)")
 
print(f"\n[Jumlah context_spans per record]")
span_count_dist = Counter(span_counts)
for n_span in sorted(span_count_dist):
    c = span_count_dist[n_span]
    print(f"  {n_span} span: {c:>6} ({c/len(records)*100:.1f}%)")
 
very_short_inst = [r for r in records if len(r["instruction"].split()) < 5]
very_short_resp = [r for r in records if len(r["response"].split()) < 6]
very_short_cot  = [r for r in records if len(r.get("cot","").split()) < 5]
print(f"\nInstruction < 5 kata  : {len(very_short_inst)}")
print(f"Response < 6 kata     : {len(very_short_resp)}")
print(f"CoT < 5 kata          : {len(very_short_cot)}")
 
report["statistik_panjang"] = {
    "instruction": stat_inst, "response": stat_resp, "cot": stat_cot,
    "span_count_distribution": dict(span_count_dist),
    "n_instruction_sangat_pendek": len(very_short_inst),
    "n_response_sangat_pendek": len(very_short_resp),
    "n_cot_sangat_pendek": len(very_short_cot),
}


[Instruction (kata)]
  min=4 max=67 mean=18.8 median=18
  [   0–   5 kata):      1 (0.0%)
  [   5–  10 kata):    497 (4.1%)
  [  10–  20 kata):   6878 (56.1%)
  [  20–  50 kata):   4875 (39.8%)
  [  50– 100 kata):     10 (0.1%)

[Response (kata)]
  min=4 max=277 mean=34.8 median=30
  [   0–   5 kata):      1 (0.0%)
  [   5–  10 kata):    257 (2.1%)
  [  10–  20 kata):   2356 (19.2%)
  [  20–  50 kata):   7475 (61.0%)
  [  50– 100 kata):   2009 (16.4%)
  [ 100– 200 kata):    159 (1.3%)
  [ 200–9999 kata):      4 (0.0%)

[CoT (kata)]
  min=10 max=173 mean=38.3 median=35
  [  10–  20 kata):    468 (3.8%)
  [  20–  50 kata):   9942 (81.1%)
  [  50– 100 kata):   1654 (13.5%)
  [ 100– 200 kata):    197 (1.6%)

[Jumlah context_spans per record]
  1 span:   8268 (67.4%)
  2 span:   3030 (24.7%)
  3 span:    963 (7.9%)

Instruction < 5 kata  : 1
Response < 6 kata     : 8
CoT < 5 kata          : 0


## 5. Pattern sumber

In [6]:
meta_patterns = {
    "awalan_teks_sumber":   r"^(berdasarkan|menurut|sesuai|dalam)\s+teks sumber",
    "awalan_konteks":       r"^(berdasarkan|menurut|sesuai|dalam)\s+konteks",
    "awalan_passages":      r"^(berdasarkan|menurut|sesuai|dalam)\s+passages",
    "awalan_dokumen_ini":   r"^(berdasarkan|menurut|sesuai|dalam)\s+dokumen ini",
    "sebut_teks_sumber_ini":r"teks sumber ini",
    "sebut_passages":       r"\bpassages\b",
    "sebut_bagian_ini":     r"bagian ini",
    "sebut_di_atas":        r"\bdi atas\b",
    "sebut_konteks_ini":    r"konteks ini",
}
 
meta_found = defaultdict(list)
for r in records:
    text = f"{r['instruction']} {r['response']}".lower()
    for name, pat in meta_patterns.items():
        if re.search(pat, text, re.IGNORECASE):
            meta_found[name].append(r)
 
for name, recs in sorted(meta_found.items(), key=lambda x: -len(x[1])):
    print(f"\n[{name}]: {len(recs)} records ({len(recs)/len(records)*100:.1f}%)")
    sample = rng.sample(recs, min(2, len(recs)))
    for r in sample:
        print(f"  [{r['subtask_type']}]")
        print(f"  Q: {r['instruction'][:120]}")
        print(f"  A: {r['response'][:200]}")
 
meta_total = len(set(id(r) for recs in meta_found.values() for r in recs))
report["meta_reference_leakage"] = {
    "n_total": meta_total,
    "pct_total": round(meta_total/len(records)*100, 2),
    "per_pattern": {k: len(v) for k, v in meta_found.items()},
}


[sebut_di_atas]: 146 records (1.2%)
  [definition_clarification]
  Q: Bagaimana Undang-Undang Nomor 28 Tahun 1999 mendefinisikan perbuatan nepotisme oleh penyelenggara negara?
  A: Menurut Undang-Undang Republik Indonesia Nomor 28 Tahun 1999, perbuatan nepotisme adalah perbuatan penyelenggara negara secara melawan hukum yang menguntungkan kepentingan keluarganya dan atau kroniny
  [factual_retrieval]
  Q: Bagaimana persentase responden yang mewajarkan tindakan mengajak anggota keluarga dalam kampanye demi mendapatkan lebih 
  A: Berdasarkan tingkat pendidikan pada tahun 2024, responden berpendidikan di bawah SMA memiliki persentase pewajaran tertinggi sebesar 17,18 persen, kelompok SMA sebesar 10,69 persen, dan kelompok di at

[sebut_bagian_ini]: 14 records (0.1%)
  [trend_interpretation]
  Q: Bagian tanaman apa saja yang umumnya digunakan sebagai sayur pada tanaman sayuran semusim, dan bagaimana pola pemanfaata
  A: Bagian tanaman yang umumnya digunakan sebagai sayur pada tanaman say

## 6. Residu

In [7]:
TABLE_RESIDUE_PATTERN = re.compile(
    r'\[\d+\]'
    r'|\b\d{1,3}(?:\.\d{3})*(?:,\d+)?\s+\d{1,3}(?:\.\d{3})*(?:,\d+)?\s+-?\d{1,3}(?:\.\d{3})*(?:,\d+)?\b'
)
 
table_residue_records = []
for r in records:
    spans = r.get("context", [])
    for span in spans:
        if TABLE_RESIDUE_PATTERN.search(span):
            table_residue_records.append((r, span))
            break
 
print(f"\n[Residu Notasi Tabel dalam context_spans]")
n_residue = len(table_residue_records)
print(f"  Record terindikasi: {n_residue} ({n_residue/len(records)*100:.1f}%)")
print("\n  Sample (3 contoh):")
for r, span in rng.sample(table_residue_records, min(3, len(table_residue_records))):
    print(f"\n  [{r['subtask_type']}] {r['record_id']}")
    print(f"  span bermasalah: {span[:150]}")
 
report["residu_tabel"] = {
    "n_terindikasi": n_residue,
    "pct_terindikasi": round(n_residue/len(records)*100, 2),
}


[Residu Notasi Tabel dalam context_spans]
  Record terindikasi: 1915 (15.6%)

  Sample (3 contoh):

  [factual_retrieval] 5990cb6e21067662
  span bermasalah: Sulawesi Utara 1,49 0,28 18,71 0,94 2,03 41,05 1,18 2,88 38,73 43,37 11,72 0,76 6,47 10,23 13,21

  [trend_interpretation] e55a517b1e9f5de3
  span bermasalah: Papua Barat Daya NA – 54,852 – – 4,93 0,73 14,89 3,49 6,37 70,30 1,15 1,64 68,04 72,56 Papua 0,43 0,16 36,191 0,13 0,74 4,95 0,64 12,93 3,69 6,20 2,82

  [document_overview] cd3dc8127c355266
  span bermasalah: Indonesia 23,67 45,40 14,46 3,29 19,49 4,92 3,14 4,97 1,58 0,39 14,44


## 7. Angka tidak terverifikasi

In [8]:
NUM_ID = re.compile(r"\b\d{1,3}(?:\.\d{3})*(?:,\d+)?\b|\b\d+,\d+\b|\b\d{2,}\b")
 
def check_numbers(r):
    nums = NUM_ID.findall(r["response"])
    if not nums:
        return [], []
    ctx_joined = r.get("context_joined") or "\n\n---\n\n".join(r.get("context", []))
    unmatched = [n for n in nums if n not in ctx_joined]
    return nums, unmatched
 
no_match_count = 0
sample_unmatched = []
for r in records:
    nums, unmatched = check_numbers(r)
    if unmatched:
        no_match_count += 1
        if len(sample_unmatched) < 5:
            sample_unmatched.append((r, nums, unmatched))
 
print(f"\nRecord dengan angka tak cocok ke context: {no_match_count} ({no_match_count/len(records)*100:.1f}%)")
print("\nSample (5 contoh):")
for r, nums, unmatched in sample_unmatched:
    print(f"\n  [{r['subtask_type']}]")
    print(f"  Q: {r['instruction'][:100]}")
    print(f"  A: {r['response'][:200]}")
    print(f"  Semua angka di response: {nums[:10]}")
    print(f"  Tidak cocok di context : {unmatched[:5]}")
 
report["kesesuaian_numerik"] = {
    "n_tidak_cocok": no_match_count,
    "pct_tidak_cocok": round(no_match_count/len(records)*100, 2),
}


Record dengan angka tak cocok ke context: 3015 (24.6%)

Sample (5 contoh):

  [methodology_explanation]
  Q: Bagaimana cara mengukur penyuapan pada lingkungan bisnis menurut Indikator SDGs 16.5.2?
  A: Pengukuran pada Indikator SDGs 16.5.2 dilakukan dengan menghitung seberapa sering perusahaan diminta atau terlibat dalam penyuapan saat berinteraksi dengan pemerintah, seperti ketika mengajukan tender
  Semua angka di response: ['16', '5', '2']
  Tidak cocok di context : ['16', '5', '2']

  [causal_reasoning]
  Q: Apa dampak yang disebutkan terjadi jika terjadi penurunan pada Indikator SDGs 16.5.1 yang mengukur p
  A: Penurunan pada Indikator SDGs 16.5.1 menunjukkan bahwa reformasi birokrasi, peningkatan transparansi, dan efisiensi layanan publik telah berhasil memutus rantai permintaan suap oleh pejabat, sehingga 
  Semua angka di response: ['16', '5', '1']
  Tidak cocok di context : ['16', '5', '1']

  [causal_reasoning]
  Q: Apa konsekuensi yang dijelaskan jika Indikator SDGs 16.5.2 

## 8. Mengukur RAFT-alignment dataset

In [9]:
def cot_quotes_verified(r):
    cot = r.get("cot", "")
    ctx_joined = r.get("context_joined") or "\n\n---\n\n".join(r.get("context", []))
    ctx_norm = " ".join(ctx_joined.split()).lower()
    quotes = re.findall(r'"([^"]{6,})"', cot)
    if not quotes:
        return None  # cot tidak memakai kutip literal, tidak bisa diverifikasi programatik
    verified = [q for q in quotes if " ".join(q.split()).lower() in ctx_norm]
    return len(verified) == len(quotes)
 
verif_results = [cot_quotes_verified(r) for r in records]
n_no_quote = sum(1 for v in verif_results if v is None)
n_verified = sum(1 for v in verif_results if v is True)
n_failed   = sum(1 for v in verif_results if v is False)
 
print(f"\n[Verifikasi Kutipan CoT terhadap context_spans]")
print(f"  CoT tanpa kutip literal (tidak terverifikasi programatik) : {n_no_quote} ({n_no_quote/len(records)*100:.1f}%)")
print(f"  CoT dengan kutip, TERVERIFIKASI cocok context             : {n_verified} ({n_verified/len(records)*100:.1f}%)")
print(f"  CoT dengan kutip, TIDAK cocok context (berpotensi mengarang): {n_failed} ({n_failed/len(records)*100:.1f}%)")
 
report["verifikasi_cot"] = {
    "n_tanpa_kutip_literal": n_no_quote,
    "n_terverifikasi": n_verified,
    "n_gagal_verifikasi": n_failed,
    "pct_gagal_dari_yang_berkutip": round(n_failed/(n_verified+n_failed)*100, 2) if (n_verified+n_failed) else None,
}


[Verifikasi Kutipan CoT terhadap context_spans]
  CoT tanpa kutip literal (tidak terverifikasi programatik) : 7747 (63.2%)
  CoT dengan kutip, TERVERIFIKASI cocok context             : 4092 (33.4%)
  CoT dengan kutip, TIDAK cocok context (berpotensi mengarang): 422 (3.4%)


## 9. konsistensi terminologi

In [10]:
TERMINOLOGY_MAP = {
    "TPT": ["tpt", "tingkat pengangguran terbuka"],
    "IPM": ["ipm", "indeks pembangunan manusia"],
    "PDRB": ["pdrb", "produk domestik regional bruto"],
    "Susenas": ["susenas", "survei sosial ekonomi nasional"],
    "garis kemiskinan": ["garis kemiskinan"],
    "TPAK": ["tpak", "tingkat partisipasi angkatan kerja"],
    "AHH": ["ahh", "angka harapan hidup"],
    "HLS": ["hls", "harapan lama sekolah"],
    "RLS": ["rls", "rata-rata lama sekolah"],
}
 
def term_coverage(records, term_map):
    usage = {k: 0 for k in term_map}
    mismatch_records = []
    for r in records:
        ctx_joined = (r.get("context_joined") or "\n\n---\n\n".join(r.get("context", []))).lower()
        resp = r.get("response", "").lower()
        for term, variants in term_map.items():
            if any(re.search(r'\b'+re.escape(v)+r'\b', ctx_joined) for v in variants):
                usage[term] += 1
            resp_has = any(re.search(r'\b'+re.escape(v)+r'\b', resp) for v in variants)
            ctx_has  = any(re.search(r'\b'+re.escape(v)+r'\b', ctx_joined) for v in variants)
            if resp_has and not ctx_has:
                mismatch_records.append((r["record_id"], term))
    return usage, mismatch_records
 
term_usage, term_mismatches = term_coverage(records, TERMINOLOGY_MAP)
print(f"\n[Kemunculan Istilah Teknis Kunci di context_spans]")
for k, v in sorted(term_usage.items(), key=lambda x: -x[1]):
    print(f"  {k:<20}: {v:>4} record")
 
print(f"\n[Ketidakcocokan Terminologi \u2014 response memakai istilah tanpa dukungan context]")
print(f"  Jumlah kasus: {len(term_mismatches)}")
for rid, term in term_mismatches[:5]:
    print(f"    {rid}  ->  {term}")
 
report["konsistensi_terminologi"] = {
    "cakupan_kamus_istilah": len(TERMINOLOGY_MAP),
    "kemunculan_per_istilah": term_usage,
    "n_mismatch": len(term_mismatches),
    "pct_mismatch": round(len(term_mismatches)/len(records)*100, 2),
}


[Kemunculan Istilah Teknis Kunci di context_spans]
  Susenas             :  764 record
  IPM                 :  257 record
  RLS                 :  125 record
  garis kemiskinan    :   81 record
  HLS                 :   64 record
  TPT                 :   39 record
  PDRB                :   38 record
  TPAK                :   18 record
  AHH                 :   16 record

[Ketidakcocokan Terminologi — response memakai istilah tanpa dukungan context]
  Jumlah kasus: 331
    c308124c97b9a450  ->  HLS
    3bb2df3055ad2a1d  ->  IPM
    98a11455f6c995ac  ->  TPT
    d1db2a2a5288cdd2  ->  IPM
    287cbcaf19fb3557  ->  PDRB


## 10. Kebocoran aturan

In [11]:
thinking_patterns = {
    "tidak_disebut_eksplisit": r"\(tidak disebut eksplisit",
    "dihitung_dari":           r"\(dihitung dari",
    "estimasi":                r"\(estimasi\)",
    "kemungkinan":             r"\(kemungkinan\)",
    "diasumsikan":             r"\(diasumsikan",
    "perkiraan":               r"\bperkiraan\)",
}
 
thinking_found = defaultdict(list)
for r in records:
    resp = r["response"].lower()
    for name, pat in thinking_patterns.items():
        if re.search(pat, resp, re.IGNORECASE):
            thinking_found[name].append(r)
 
for name, recs in sorted(thinking_found.items(), key=lambda x: -len(x[1])):
    print(f"\n[{name}]: {len(recs)} records")
    for r in rng.sample(recs, min(2, len(recs))):
        print(f"  [{r['subtask_type']}] A: {r['response'][:250]}")
 
thinking_total = len(set(id(r) for recs in thinking_found.values() for r in recs))
report["thinking_artifacts"] = {
    "n_total": thinking_total,
    "pct_total": round(thinking_total/len(records)*100, 2),
    "per_pattern": {k: len(v) for k, v in thinking_found.items()},
}

## 7. Duplikat Intruksi/pertanyaan

In [12]:
norm_inst = Counter(
    re.sub(r"[^\w\s]", "", r["instruction"].lower()).strip()
    for r in records
)
exact_dups = {k: v for k, v in norm_inst.items() if v > 1}
print(f"\nInstruction exact duplicate: {len(exact_dups)} cluster, "
      f"{sum(exact_dups.values())} records total")
 
print("\nTop 5 instruction yang paling sering muncul:")
for inst, count in sorted(exact_dups.items(), key=lambda x: -x[1])[:5]:
    print(f"  ({count}x) {inst[:100]}")
 
report["duplikasi_exact"] = {
    "n_cluster": len(exact_dups),
    "n_records_involved": sum(exact_dups.values()),
}


Instruction exact duplicate: 49 cluster, 106 records total

Top 5 instruction yang paling sering muncul:
  (4x) bagaimana cara menghitung laju pertumbuhan produk domestik bruto pdb
  (4x) uraikan gambaran sampling error persentase perempuan pernah kawin berumur 1549 tahun yang menggunaka
  (4x) uraikan gambaran sampling error persentase penduduk yang mempunyai keluhan kesehatan dalam sebulan t
  (3x) apa yang dimaksud dengan relative standard error rse dalam tabel sampling error
  (3x) apa yang dimaksud dengan relative standard error rse dalam tabel ini


## 8. Pertanyaan mirip jawaban

In [13]:
import difflib
high_sim = []
for r in records:
    instr = r["instruction"].lower()
    resp  = r["response"].lower()
    sim = difflib.SequenceMatcher(None, instr, resp).ratio()
    r["_sim"] = sim
    if sim > 0.85:
        high_sim.append(r)
 
print(f"\nSimilarity > 0.85: {len(high_sim)} records ({len(high_sim)/len(records)*100:.1f}%)")
print("\nSample (perhatikan apakah ini legitimate untuk definition_clarification):")
by_subtask = defaultdict(list)
for r in high_sim:
    by_subtask[r["subtask_type"]].append(r)
for subtask, recs in sorted(by_subtask.items(), key=lambda x: -len(x[1]))[:4]:
    print(f"\n  [{subtask}] \u2014 {len(recs)} records")
    r = recs[0]
    print(f"  Q: {r['instruction'][:120]}")
    print(f"  A: {r['response'][:200]}")
    print(f"  sim={r['_sim']:.3f}")
 
report["instruction_response_similarity"] = {
    "n_high_sim": len(high_sim),
    "pct_high_sim": round(len(high_sim)/len(records)*100, 2),
}


Similarity > 0.85: 326 records (2.7%)

Sample (perhatikan apakah ini legitimate untuk definition_clarification):

  [factual_retrieval] — 235 records
  Q: Menurut Global Corruption Barometer 2020, berapa persen pengguna layanan publik di Indonesia yang mengaku membayar suap 
  A: Menurut Global Corruption Barometer 2020, 30 persen pengguna layanan publik di Indonesia mengaku telah membayar suap dalam kurun waktu 12 bulan sebelum survei.
  sim=0.893

  [source_navigation] — 86 records
  Q: Pada gambar nomor berapa data mengenai persentase pelaku usaha yang pernah berurusan dengan layanan publik dan mengeluar
  A: Data persentase pelaku usaha yang pernah berurusan dengan layanan publik dan mengeluarkan uang, barang, atau fasilitas melebihi ketentuan periode 2020–2024 ditampilkan pada Gambar 5.3.
  sim=0.865

  [definition_clarification] — 2 records
  Q: Menurut Pasal 1 angka 14 Undang-Undang Republik Indonesia Nomor 20 Tahun 2003 tentang Sistem Pendidikan Nasional, bagaim
  A: Menurut P

## 9. Validitas pertanyaan/intruksi

In [14]:
ALLOWED_TASKS = {"qa", "summary"}
ALLOWED_SUBTASKS = {
    "factual_retrieval", "source_navigation",
    "definition_clarification", "methodology_explanation",
    "trend_interpretation", "causal_reasoning",
    "comparative_analysis", "multi_source_synthesis",
    "document_overview",
}
invalid_task    = [r for r in records if r.get("task_type") not in ALLOWED_TASKS]
invalid_subtask = [r for r in records if r.get("subtask_type") not in ALLOWED_SUBTASKS]
missing_fields  = [r for r in records if not r.get("instruction") or not r.get("response")
                   or not r.get("context") or not r.get("context_chunk_ids")
                   or not r.get("cot")]
 
print(f"\nInvalid task_type    : {len(invalid_task)}")
print(f"Invalid subtask_type : {len(invalid_subtask)}")
print(f"Missing required field (termasuk cot): {len(missing_fields)}")
 
report["validasi_skema"] = {
    "n_invalid_task": len(invalid_task),
    "n_invalid_subtask": len(invalid_subtask),
    "n_missing_fields": len(missing_fields),
}


Invalid task_type    : 0
Invalid subtask_type : 0
Missing required field (termasuk cot): 0


## 10. Ringkasan

In [15]:
total = len(records)
print(f"\nTotal records: {total}")
 
print(f"\n[Calon HARD GATE \u2014 buang]")
print(f"  schema invalid (task/subtask)       : {len(invalid_task)+len(invalid_subtask)}")
print(f"  missing required fields             : {len(missing_fields)}")
print(f"  response < 6 kata                   : {len(very_short_resp)}")
print(f"  cot < 5 kata / kosong                : {len(very_short_cot)}")
print(f"  residu notasi tabel dalam span       : {n_residue}")
print(f"  thinking artifacts (estimasi, dll)   : {thinking_total}")
 
print(f"\n[Calon CLEANING \u2014 strip, bukan buang]")
print(f"  meta-reference awalan               : {meta_total} ({meta_total/total*100:.1f}%)")
 
print(f"\n[Calon SOFT SCORE \u2014 penalti tapi tidak buang]")
print(f"  angka tidak terverifikasi           : {no_match_count} ({no_match_count/total*100:.1f}%)")
print(f"  cot gagal verifikasi kutipan        : {n_failed} ({n_failed/total*100:.1f}%)")
print(f"  terminology mismatch                : {len(term_mismatches)} ({len(term_mismatches)/total*100:.1f}%)")
print(f"  high similarity inst\u2248resp           : {len(high_sim)} ({len(high_sim)/total*100:.1f}%)")
 
print(f"\n[Calon NEAR-DUPLICATE \u2014 perlu embedding check]")
print(f"  exact duplicate instruction         : {sum(exact_dups.values())} (minimum, sebelum semantic dedup)")
 
print(f"\n[Cakupan Korpus]")
print(f"  dokumen unik                        : {n_docs}")
print(f"  section unik                        : {len(section_counts)}")
 
report["ringkasan_final"] = {
    "total_records": total,
    "estimasi_hard_gate": {
        "schema_invalid": len(invalid_task)+len(invalid_subtask),
        "missing_fields": len(missing_fields),
        "response_terlalu_pendek": len(very_short_resp),
        "cot_terlalu_pendek": len(very_short_cot),
        "residu_tabel": n_residue,
        "thinking_artifacts": thinking_total,
    },
    "estimasi_soft_flag": {
        "angka_tidak_cocok": no_match_count,
        "cot_gagal_verifikasi": n_failed,
        "terminology_mismatch": len(term_mismatches),
        "high_similarity": len(high_sim),
    },
    "estimasi_duplikat_exact": sum(exact_dups.values()),
}
 
with open(f"{OUTPUT_DIR}/laporan_eksplorasi_bab4.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
 
with open(f"{OUTPUT_DIR}/meta_reference_samples.jsonl", "w") as f:
    for recs in meta_found.values():
        for r in recs:
            f.write(json.dumps({"record_id": r["record_id"], "subtask": r["subtask_type"],
                                "instruction": r["instruction"], "response": r["response"]},
                               ensure_ascii=False) + "\n")
 
with open(f"{OUTPUT_DIR}/thinking_artifact_samples.jsonl", "w") as f:
    for recs in thinking_found.values():
        for r in recs:
            f.write(json.dumps({"record_id": r["record_id"], "subtask": r["subtask_type"],
                                "response": r["response"]}, ensure_ascii=False) + "\n")
 
with open(f"{OUTPUT_DIR}/table_residue_samples.jsonl", "w") as f:
    for r, span in table_residue_records:
        f.write(json.dumps({"record_id": r["record_id"], "subtask": r["subtask_type"],
                            "span": span}, ensure_ascii=False) + "\n")
 
print(f"\n[Laporan lengkap Bab 4 disimpan ke {OUTPUT_DIR}/laporan_eksplorasi_bab4.json]")
print("Semua angka di atas siap dikutip langsung sebagai tabel/paragraf eksplorasi dataset.")


Total records: 12261

[Calon HARD GATE — buang]
  schema invalid (task/subtask)       : 0
  missing required fields             : 0
  response < 6 kata                   : 8
  cot < 5 kata / kosong                : 0
  residu notasi tabel dalam span       : 1915
  thinking artifacts (estimasi, dll)   : 0

[Calon CLEANING — strip, bukan buang]
  meta-reference awalan               : 183 (1.5%)

[Calon SOFT SCORE — penalti tapi tidak buang]
  angka tidak terverifikasi           : 3015 (24.6%)
  cot gagal verifikasi kutipan        : 422 (3.4%)
  terminology mismatch                : 331 (2.7%)
  high similarity inst≈resp           : 326 (2.7%)

[Calon NEAR-DUPLICATE — perlu embedding check]
  exact duplicate instruction         : 106 (minimum, sebelum semantic dedup)

[Cakupan Korpus]
  dokumen unik                        : 29
  section unik                        : 634

[Laporan lengkap Bab 4 disimpan ke output_05b_exploration/laporan_eksplorasi_bab4.json]
Semua angka di atas siap dikut